In [1]:
from llama_index.core import Settings, StorageContext, VectorStoreIndex, Document, SimpleDirectoryReader
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.agent import AgentWorkflow
from llama_index.core.workflow import Context
from llama_index.llms.groq import Groq
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

Settings.llm = Groq(model="openai/gpt-oss-20b",
                    context_window=8000)

path = "./models/nomic-ai-v1.5"
Settings.embed_model = HuggingFaceEmbedding(
    model_name=path,
    trust_remote_code=True
)

/home/arthas/anaconda3/envs/tcc/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1778439691.268755   76257 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778439691.298598   76257 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
E0000 00:00:1778439693.713351   76257 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than 

In [6]:
db = chromadb.PersistentClient(path="./chroma_db")
collection = db.get_or_create_collection("recipes")
vector_store = ChromaVectorStore(chroma_collection=collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

if collection.count() == 0:
    documents = SimpleDirectoryReader(
        input_files=["./data/some_recipes.csv"]
    ).load_data()
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context
    )

else:
    index = VectorStoreIndex.from_vector_store(
        vector_store=vector_store,
        storage_context=storage_context
    )

query_engine = index.as_query_engine()

async def find_recipes(query: str) -> str:
    """useful to search for recipes in the database"""
    response = await query_engine.aquery(query)
    return str(response)

agent = AgentWorkflow.from_tools_or_functions(
    [find_recipes],
    llm=Settings.llm,
    system_prompt="""You are a kitchen chef. Use ONLY the provided tools for information.
                     Do not respond anything that isn't in the document"""
)
async def main():
    ctx = Context(agent)
    response = await agent.run("Say me about chile rellenos", ctx=ctx)
    print(response)
    response = await agent.run("How much steps its take long and what is it's id?", ctx=ctx)
    print(response)

await main()

**Crispy, Flaky Chile Relleno**

- **Ingredients**: egg‑roll wrappers, whole green chilies, cheese, cornstarch, oil  
- **Rating**: 5 stars  

This version turns the classic chile relleno into a crunchy, flaky dish. The chilies are wrapped in egg‑roll wrappers, coated with cornstarch, and fried in oil until golden and crisp. Inside, the cheese melts to a creamy center, giving each bite a satisfying contrast of textures.
I’m sorry, but I don’t have that information.


In [11]:
from llama_index.core.node_parser import SentenceSplitter

documents = SimpleDirectoryReader(
        input_files=["./data/some_recipes.csv"]
    ).load_data()
parser = SentenceSplitter(chunk_size=256, chunk_overlap=30)
nodes = parser.get_nodes_from_documents(documents=documents)

for i, node in enumerate(nodes[:5]):
    # It makes sense... the file's split is terrible
    print(f"--- Node {i} (Size: {len(node.get_content())} characters) ---")
    print(node.get_content())
    print(f"Metadata: {node.metadata}\n")

--- Node 0 (Size: 786 characters) ---
137739, arriba   baked winter squash mexican style, 55, 47892, 2005-09-16, ['60-minutes-or-less', 'time-to-make', 'course', 'main-ingredient', 'cuisine', 'preparation', 'occasion', 'north-american', 'side-dishes', 'vegetables', 'mexican', 'easy', 'fall', 'holiday-event', 'vegetarian', 'winter', 'dietary', 'christmas', 'seasonal', 'squash'], [51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0], 11, ['make a choice and proceed with recipe', 'depending on size of squash , cut into half or fourths', 'remove seeds', 'for spicy squash , drizzle olive oil or melted butter over each cut squash piece', 'season with mexican seasoning mix ii', 'for sweet squash , drizzle melted honey , butter , grated piloncillo over each cut squash piece', 'season with sweet mexican spice mix', 'bake at 350 degrees ,
Metadata: {'file_path': 'data/some_recipes.csv', 'file_name': 'some_recipes.csv', 'file_type': 'text/csv', 'file_size': 6468384, 'creation_date': '2026-05-10', 'last_modified_

In [14]:
import pandas as pd

def get_recipes_csv(path: str):
    df = pd.read_csv(path)
    documents = []
    
    for _, row in df.iterrows():
        content = "\n".join([f"{col}: {val}" for col, val in row.items()])
        metadata = {"recipe_name": row.get("name")}

        documents.append(Document(text=content, metadata=metadata))
    return documents

collection = db.get_or_create_collection("recipes-parsed")
vector_store = ChromaVectorStore(chroma_collection=collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

if collection.count() == 0:
    documents = get_recipes_csv("./data/some_recipes.csv")
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context
    )

else:
    index = VectorStoreIndex.from_vector_store(
        vector_store=vector_store,
        storage_context=storage_context
    )

query_engine = index.as_query_engine()
async def find_recipes_correctly(query: str) -> str:
    """useful to search for recipes in the database"""
    response = await query_engine.aquery(query)
    return str(response)

agent = AgentWorkflow.from_tools_or_functions(
    [find_recipes_correctly],
    llm=Settings.llm,
    system_prompt="""You are a kitchen chef. Use ONLY the provided tools for information.
                     Do not respond anything that isn't in the document"""
)
async def main():
    ctx = Context(agent)
    response = await agent.run("Say me about chile rellenos", ctx=ctx)
    print(response)
    response = await agent.run("How much steps it takes and what is it's id?", ctx=ctx)
    print(response)

await main()

2026-05-10 17:00:12,147 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-10 17:00:13,069 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-10 17:00:13,296 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


**Chile Rellenos**

*Preparation time:* 45 minutes  
*Servings:* 5 ingredients, suitable for 4–6 people  
*Cuisine:* Southwestern United States, American

**Ingredients**  
- Egg roll wrappers  
- Whole green chilies (drained)  
- Cheese (any firm variety, cut into sticks)  
- Cornstarch  
- Oil (for deep‑frying)

**Method**  
1. Drain the green chilies.  
2. Sprinkle cornstarch on a sheet pan.  
3. Open a chile, place it on an egg‑roll wrapper with the corner facing you.  
4. Add a cheese stick to the chile, roll up, tuck in the corners, and moisten the edges to seal.  
5. Toss the rolled chile in cornstarch, dusting generously.  
6. Shake off excess cornstarch and deep‑fry in oil at 350 °F until lightly golden and the cheese just starts to ooze.  
7. Drain on paper towels and keep warm in a 250 °F oven.  
8. Serve as a side dish or appetizer with your favorite Mexican meal.  
9. Add your favorite condiments or sauces.


2026-05-10 17:00:14,026 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-10 17:00:15,212 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-10 17:00:15,398 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-10 17:00:16,554 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-10 17:00:16,598 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-05-10 17:00:16,599 - INFO - Retrying request to /chat/completions in 9.000000 seconds
2026-05-10 17:00:25,896 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


It has 9 steps.


In [26]:
# The real information
pd.set_option('display.max_colwidth', None)
df = pd.read_csv("./data/some_recipes.csv")
df.loc[df["id"] == 43026]

,id,name,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
14,43026,chile rellenos,45,52268,2002-10-14,"['60-minutes-or-less', 'time-to-make', 'course', 'main-ingredient', 'cuisine', 'preparation', 'north-american', '5-ingredients-or-less', 'appetizers', 'side-dishes', 'eggs-dairy', 'american', 'southwestern-united-states', 'easy', 'vegetarian', 'cheese', 'deep-fry', 'dietary', 'oamc-freezer-make-ahead', 'number-of-servings', 'technique']","[94.0, 10.0, 0.0, 11.0, 11.0, 21.0, 0.0]",9,"['drain green chiles', 'sprinkle cornstarch on sheet pan', 'open up chile and place on egg roll wrapper with corner facing you', 'place cheese stick on chile and roll up , tucking in corners and moistening edges to seal', 'toss in cornstarch , dusting liberally', 'shake off excess cornstarch and deep fry in 350 oil to cover until lightly golden and cheese just starts to ooze out', 'drain on paper towels and keep warm in 250 oven', 'serve as a side dish or appetizer with your favorite mexican meal', 'add your favorite condi- ments / sauces']","a favorite from a local restaurant no longer in business. not an authentic mexican chile relleno, but a crispy, flaky version.","['egg roll wrap', 'whole green chilies', 'cheese', 'cornstarch', 'oil']",5
